# Aula 5 — Métricas, exploração e hashtags

Na Aula 4 a gente coletou dados de rede social e deu uma primeira olhada neles: quantas linhas, quais colunas, vídeo mais curtido, autor mais frequente. Hoje a gente troca de ferramenta e de profundidade: em vez de `csv.DictReader` e listas de dicionários, usamos o **Pandas**, que é a biblioteca padrão pra trabalhar com dados em tabela no Python.

O motivo de trocar de ferramenta agora é simples: as perguntas de hoje (agrupar por hashtag, calcular taxa de engajamento, comparar autores) dão muito mais trabalho fazendo na mão do que com Pandas, que já vem com essas operações prontas. A aula segue um fio único: carregar os dados, conferir a qualidade deles, entender métricas de rede social (e por que denominador importa tanto quanto numerador), calcular estatística descritiva, filtrar, agrupar por hashtag e fechar com uma tabela-resumo. Essa tabela-resumo é a base do Projeto 2.

## 1. Revisão rápida: linhas, colunas, tipos e valores ausentes

Só pra alinhar o vocabulário antes de seguir (já visto nas Aulas 2 e 4):

- **Linha** é um registro, um post/vídeo coletado.
- **Coluna** é uma variável, um campo do post (`likes`, `author`, `hashtags` etc).
- **Tipo de dado** é a natureza do valor guardado numa coluna: número inteiro, texto, data, verdadeiro/falso. O Pandas tenta adivinhar o tipo de cada coluna sozinho ao ler o CSV, e às vezes erra (por exemplo, uma coluna de números pode virar texto se tiver algum valor vazio ou fora do padrão).
- **Valor ausente** é quando uma célula da tabela não tem informação (a plataforma não preencheu aquele campo pra aquele post). O Pandas representa isso como `NaN` (Not a Number), mesmo em colunas de texto.

Essas quatro ideias voltam o tempo todo hoje, então se alguma ficou vaga, esse é o momento de perguntar.

## 2. Preparando o ambiente

Esta aula usa só o `pandas` (sem `wordcloud` nem `matplotlib`: visualização de verdade é assunto da Aula 6).

No terminal (o integrado do VS Code funciona bem aqui), **dentro da pasta da disciplina** (onde já existe o `.venv` que você criou antes; não precisa criar outro):

```cmd
uv pip install -r requirements.txt
```

Se o `uv` não funcionar:

```cmd
pip install -r requirements.txt
```

Os mesmos comandos funcionam no Mac (Terminal).

**OBS:** no VS Code/Cursor, confira se o kernel do notebook está apontando para o `.venv` da raiz do repositório, não para outro Python. Qualquer coisa, só clicar em "Select Kernel".


## 3. Carregando a exportação com Pandas

`dados/exportacao.csv` é o mesmo tipo de arquivo da Aula 4: um export do Zeeschuimer, separado por `;`. A diferença de hoje é que, em vez de abrir o arquivo linha por linha, pedimos pro Pandas ler tudo de uma vez e organizar como uma tabela, chamada de **DataFrame**.

**OBS:** o arquivo desta pasta é uma exportação **real** do Zeeschuimer (vários perfis do TikTok misturados: g1, CNN, Botafogo, CazéTV etc), pra todo mundo rodar a aula com o mesmo material. No exercício, você troca pelo arquivo da sua própria coleta da Aula 4.

In [13]:
import pandas as pd  # importa o pandas com o apelido "pd", convenção usada por quase todo mundo

df = pd.read_csv("dados/exportacao.csv", sep=";")  # lê o CSV inteiro de uma vez, organizado numa tabela (DataFrame), avisando que o separador é ";"

print(f"Linhas: {df.shape[0]}, colunas: {df.shape[1]}")  # shape devolve uma dupla (linhas, colunas); aqui só mostramos os dois números

Linhas: 640, colunas: 36


In [14]:
df[["author", "body", "likes", "comments", "shares", "plays", "hashtags", "timestamp"]].head()  # mostra só as colunas que vamos usar na aula (o CSV tem bem mais campos)

,author,body,likes,comments,shares,plays,hashtags,timestamp
0,g1,Saúde - A Organização Mundial da Saúde (OMS) d...,0,0,0,229,"g1,vacina,oms,saúde,g1saúde,tiktoknotícias",2026-08-11 13:33:29
1,g1,Baixada Fluminense - Um vídeo mostra uma menin...,188,27,6,10600,"g1,tiktoknotícias,baixadafluminense,assaltante...",2026-08-11 13:24:19
2,g1,Rio de Janeiro - A Polícia Civil prendeu em fl...,1525,46,39,38000,"g1,riodejaneiro,veterinário,gato,animais,g1loc...",2026-08-11 13:13:34
3,g1,Saída da Turquia - O presidente dos Estados Un...,108,16,1,13000,"g1,trump,estadosunidos,airforceone,g1mundo,tik...",2026-08-11 13:07:20
4,g1,Tragédia na Colômbia - A Colômbia entra nesta ...,2010,27,20,61100,"g1,g1mundo,terremoto,colômbia,tiktoknotícias",2026-08-11 12:58:26


## 4. Conferindo a qualidade dos dados

Antes de calcular qualquer métrica, vale conferir três coisas: se os tipos de cada coluna fazem sentido, se tem valor ausente em campo importante e se tem linha duplicada (às vezes uma ferramenta de coleta captura o mesmo post duas vezes).

In [15]:
df.dtypes[["likes", "comments", "shares", "plays", "hashtags", "author", "timestamp"]]  # mostra o tipo de cada uma dessas colunas: int64 é número inteiro, object/str é texto

likes        int64
comments     int64
shares       int64
plays        int64
hashtags       str
author         str
timestamp      str
dtype: object

**O que observar:** `likes`, `comments`, `shares` e `plays` vieram como número inteiro (`int64`), ótimo, porque vamos fazer conta com eles direto. `hashtags`, `author` e `timestamp` vieram como texto, o que também é esperado: `timestamp` ainda não virou data de verdade aos olhos do Pandas, ele só está guardando a string. Se sua coleta real trouxer alguma coluna de número como texto, normalmente é sinal de que tem algum valor fora do padrão misturado ali (por exemplo, um campo vazio ou um texto onde devia ter só dígito).

In [16]:
df[["author", "body", "hashtags", "likes", "comments", "shares", "plays"]].isna().sum()  # conta, por coluna, quantas linhas têm valor ausente (NaN)

author        0
body          0
hashtags    156
likes         0
comments      0
shares        0
plays         0
dtype: int64

**O que observar:** nesta coleta, várias linhas não têm `hashtags` preenchido (o post não usou nenhuma, ou o Zeeschuimer não capturou esse campo pra ele). As colunas de curtida, comentário, compartilhamento e visualização vieram completas. Isso é normal em dado real, e vamos precisar lidar com os ausentes mais na frente, quando agruparmos por hashtag: não dá pra agrupar o que não existe.

In [17]:
duplicadas = df.duplicated().sum()  # conta quantas linhas são idênticas a outra linha anterior, considerando todas as colunas
print(f"Linhas duplicadas: {duplicadas}")

df = df.drop_duplicates()  # remove as linhas duplicadas, mantendo só a primeira ocorrência de cada uma
print(f"Linhas depois de remover duplicatas: {len(df)}")

Linhas duplicadas: 0
Linhas depois de remover duplicatas: 640


**ATENÇÃO:** duplicata aqui significa "linha idêntica em todas as colunas". Dois posts diferentes que por acaso têm o mesmo número de curtidas NÃO são duplicatas, então não se assuste se `duplicated()` der zero: é bem normal se a sua coleta não teve problema de captura repetida. É o comportamento esperado do Zeeschuimer (que pode divergir às vezes e pegar repetido, mas é raro...)

## 5. Métricas de rede social: o que cada uma quer dizer

Antes de sair calculando, vale alinhar quatro palavras que aparecem toda hora em análise de rede social, e que às vezes são usadas de forma solta demais:

- **Alcance:** quantas contas/pessoas diferentes viram o post, pelo menos uma vez. É sobre pessoas únicas, não sobre quantas vezes o post foi visto.
- **Impressões:** quantas vezes o post foi exibido no total, contando repetição (a mesma pessoa pode ver o post mais de uma vez, rolando o feed de novo). Impressões costuma ser um número maior ou igual ao alcance.
- **Engajamento:** quanto as pessoas interagiram com o post (curtida, comentário, compartilhamento), normalmente expresso como **taxa**, não como número bruto: interações dividido por algum denominador (alcance, impressões ou visualizações, dependendo do que a plataforma disponibiliza).
- **Retenção:** de quem começou a assistir um vídeo, quantos ficaram até um certo ponto (ou até o fim). É uma métrica de vídeo, calculada a partir de dados de reprodução segundo a segundo.

**ATENÇÃO:** essas quatro métricas raramente vêm prontas num export como o do Zeeschuimer. Elas costumam existir de verdade só no painel nativo da plataforma (TikTok Analytics, Meta Business Suite etc), porque dependem de dados que só o dono da conta enxerga. O que temos no nosso CSV são `likes`, `comments`, `shares` e `plays` (visualizações), e é a partir deles que vamos aproximar essas ideias.

### Por que o denominador importa tanto quanto o numerador

Essa é a parte que mais gera confusão, e o "Concluído quando" desta aula é exatamente sobre isso: **não confundir volume bruto com desempenho relativo**.

Um vídeo com 50 mil curtidas parece um sucesso enorme, só que se ele teve 5 milhões de visualizações, só 1% de quem viu curtiu. Já um vídeo com 2 mil curtidas e 10 mil visualizações teve 20% de taxa de curtida, uma proporção bem mais alta, mesmo com número absoluto menor. Sem o denominador (o `plays`, nesse caso), a curtida bruta só diz "quantas pessoas curtiram", não "o quão bem esse vídeo específico converteu quem viu em quem curtiu". A partir daqui, sempre que a gente falar em engajamento, vai ser como taxa (uma divisão), nunca como número solto.

In [18]:
# usamos "plays" (visualizações) como denominador, já que não temos alcance nem impressões no export
# antes de dividir, tiramos as linhas com plays igual a zero, pra não dividir por zero (o Python quebraria, ou devolveria infinito)
df_com_visualizacoes = df[df["plays"] > 0].copy()  # .copy() evita um aviso do pandas sobre estar alterando uma "fatia" da tabela original

print(f"Linhas descartadas por plays = 0: {len(df) - len(df_com_visualizacoes)}")

Linhas descartadas por plays = 0: 0


In [19]:
# a taxa de engajamento aproxima "quanto quem viu interagiu": soma das interações dividida pelas visualizações
df_com_visualizacoes["taxa_engajamento"] = (
    df_com_visualizacoes["likes"] + df_com_visualizacoes["comments"] + df_com_visualizacoes["shares"]
) / df_com_visualizacoes["plays"]  # a divisão inteira acontece linha a linha, automaticamente, sem precisar de laço

df_com_visualizacoes[["author", "likes", "plays", "taxa_engajamento"]].sort_values("taxa_engajamento", ascending=False).head(5)  # mostra os 5 posts com maior taxa de engajamento

,author,likes,plays,taxa_engajamento
515,botafogo,8871,34600,0.264249
507,botafogo,7817,31900,0.251505
487,botafogo,2746,11500,0.248435
630,botafogo,5456,25200,0.226349
622,botafogo,3120,14400,0.222778


In [20]:
df_com_visualizacoes.sort_values("likes", ascending=False).head(5)

,collected_from_url,id,thread_id,author,author_full,author_followers,author_likes,author_videos,author_avatar,body,...,comments,shares,plays,hashtags,challenges,diversification_labels,location_created,effects,warning,taxa_engajamento
82,https://www.tiktok.com/@g1,7672399029149388052,7672399029149388052,g1,g1,12200000,57,45600,https://p16-common-sign.tiktokcdn.com/tos-alis...,"Colômbia - Um terremoto de magnitude 7,4 ating...",...,11900,26500,16100000,"g1,colômbia,terremoto,g1mundo,tiktoknotícias","g1,colômbia,terremoto,g1mundo,tiktoknotícias",NaN,NaN,NaN,NaN,0.033248
368,https://www.tiktok.com/@cazetv?lang=pt-BR,7590182065056894216,7590182065056894216,elvizmagician,elvz,233100,5832,860,https://p19-common-sign.tiktokcdn.com/tos-alis...,2016 was 10 years ago… #lyricsvideo #coldplay ...,...,882,18800,3300000,"lyricsvideo,coldplay,edit,2016,fyp","lyricsvideo,coldplay,edit,2016,fyp",NaN,NaN,NaN,NaN,0.136782
448,https://www.tiktok.com/@cazetv?lang=pt-BR,7672202931394956560,7672202931394956560,cazetv,CazéTV,14100000,736,5285,https://p16-common-sign.tiktokcdn.com/tos-alis...,PULGAR É MAU 😡🤬 E TAMBÉM É VICIADO EM FAZER GO...,...,1496,5743,1300000,"coberturacazétv,flamengo,futbr,brasileirão,cztv12","coberturacazétv,flamengo,futbr,brasileirão,cztv12",NaN,NaN,NaN,NaN,0.177261
99,https://www.tiktok.com/@g1,7672370318475791634,7672370318475791634,g1,g1,12200000,57,45600,https://p16-common-sign.tiktokcdn.com/tos-alis...,"#Austrália - Na Austrália, uma boneca sexual r...",...,2084,27900,1900000,"austrália,g1,g1mundo,tiktoknotícias","austrália,g1,g1mundo,tiktoknotícias",NaN,NaN,NaN,NaN,0.118728
69,https://www.tiktok.com/@g1,7672421263159184660,7672421263159184660,g1,g1,12200000,57,45600,https://p16-common-sign.tiktokcdn.com/tos-alis...,Conteúdo sensível - A atriz indiana Nikita Raw...,...,3875,9972,2100000,"g1,índia,mumbai,g1pop,nikitarawal,tiktoknotícias","g1,índia,mumbai,g1pop,nikitarawal,tiktoknotícias",NaN,NaN,NaN,NaN,0.096784


**O que observar:** compare essa lista com a lista dos posts mais curtidos em número absoluto (você pode conferir com `df_com_visualizacoes.sort_values("likes", ascending=False).head(5)`). Normalmente não é a mesma lista: taxa de engajamento e curtida bruta respondem perguntas diferentes.

## 6. Estatística descritiva

Com as colunas de métrica já como número, o Pandas calcula estatística descritiva com um método só: `.describe()`. Ele devolve contagem, média, desvio padrão, mínimo, máximo e os quartis (25%, 50%, 75%) de cada coluna numérica.

In [21]:
df_com_visualizacoes[["likes", "comments", "shares", "plays"]].describe()  # estatística descritiva das quatro métricas de interação

,likes,comments,shares,plays
count,640.000000,640.000000,640.00000,6.400000e+02
mean,9902.445312,212.025000,437.86250,1.561087e+05
std,34898.873856,752.566781,2064.97749,7.289217e+05
min,0.000000,0.000000,0.00000,2.290000e+02
25%,278.500000,9.000000,6.00000,6.836000e+03
50%,1573.000000,31.500000,27.00000,2.360000e+04
75%,4920.000000,119.500000,139.00000,1.067500e+05
max,496900.000000,11900.000000,27900.00000,1.610000e+07


In [22]:
media = df_com_visualizacoes["likes"].mean()  # média: soma de tudo dividida pela quantidade de linhas
mediana = df_com_visualizacoes["likes"].median()  # mediana: o valor do meio, se você ordenasse todas as curtidas

print(f"Média de curtidas: {media:.1f}")
print(f"Mediana de curtidas: {mediana:.1f}")

Média de curtidas: 9902.4
Mediana de curtidas: 1573.0


**O que observar:** quando a média fica bem mais alta que a mediana, é sinal de que alguns poucos posts "puxam a média pra cima" (posts virais fora da curva). A mediana é mais resistente a esses valores extremos: ela conta melhor "como foi o post típico da coleta", enquanto a média conta "como foi a coleta no total, incluindo os exageros".

## 7. Filtragem de linhas

Filtrar é pedir pro Pandas devolver só as linhas que cumprem uma condição. A sintaxe é sempre parecida: `df[condição]`, onde a condição é uma comparação que devolve verdadeiro ou falso pra cada linha.

In [23]:
posts_de_um_autor = df_com_visualizacoes[df_com_visualizacoes["author"] == "g1"]  # mantém só as linhas em que a coluna "author" é exatamente esse valor (troque por outro perfil da coleta se quiser)
print(f"Posts desse autor na coleta: {len(posts_de_um_autor)}")
posts_de_um_autor[["body", "likes", "plays"]].head()  # head() pra não derramar o notebook com dezenas de textos longos

Posts desse autor na coleta: 111


,body,likes,plays
0,Saúde - A Organização Mundial da Saúde (OMS) d...,0,229
1,Baixada Fluminense - Um vídeo mostra uma menin...,188,10600
2,Rio de Janeiro - A Polícia Civil prendeu em fl...,1525,38000
3,Saída da Turquia - O presidente dos Estados Un...,108,13000
4,Tragédia na Colômbia - A Colômbia entra nesta ...,2010,61100


In [24]:
posts_populares = df_com_visualizacoes[df_com_visualizacoes["likes"] >= 5000]  # mantém só as linhas com 5000 curtidas ou mais
print(f"Posts com 5000+ curtidas: {len(posts_populares)} de {len(df_com_visualizacoes)}")

Posts com 5000+ curtidas: 154 de 640


In [25]:
# str.contains procura um pedaço de texto dentro da coluna; na=False faz as linhas sem hashtag (NaN) não darem erro na busca
posts_com_ia = df_com_visualizacoes[df_com_visualizacoes["hashtags"].str.contains("ia", na=False)]
print(f"Linhas em que 'ia' aparece em algum lugar das hashtags: {len(posts_com_ia)}")
posts_com_ia[["author", "body", "hashtags"]].head(10)

Linhas em que 'ia' aparece em algum lugar das hashtags: 203


,author,body,hashtags
0,g1,Saúde - A Organização Mundial da Saúde (OMS) d...,"g1,vacina,oms,saúde,g1saúde,tiktoknotícias"
1,g1,Baixada Fluminense - Um vídeo mostra uma menin...,"g1,tiktoknotícias,baixadafluminense,assaltante..."
2,g1,Rio de Janeiro - A Polícia Civil prendeu em fl...,"g1,riodejaneiro,veterinário,gato,animais,g1loc..."
3,g1,Saída da Turquia - O presidente dos Estados Un...,"g1,trump,estadosunidos,airforceone,g1mundo,tik..."
4,g1,Tragédia na Colômbia - A Colômbia entra nesta ...,"g1,g1mundo,terremoto,colômbia,tiktoknotícias"
5,g1,Desaceleração - O Índice Nacional de Preços ao...,"g1,tiktoknotícias,inflação,ipca,alimentosebebidas"
7,g1,China - Um foguete Longa Marcha 7A explodiu na...,"g1,china,foguete,ciência,g1ciência,tiktoknotícias"
8,g1,Botsuana - Um vídeo mostra o momento em que um...,"g1,hipopótomo,botsuana,áfrica,animais,g1meiomb..."
9,g1,Rio de Janeiro - A Justiça do Rio de Janeiro d...,"g1,anthonygarotinho,riodejaneiro,eleição2026,p..."
10,g1,#g1GuiadeCompras - É bastante comum esquecer o...,"g1guiadecompras,g1,guiadecompras,celular,tecno..."


**ATENÇÃO:** repare que `str.contains("ia")` também pega hashtags que só têm essas duas letras juntas em outro lugar (tipo `#notícias`, `#política`, `#diario`), porque ele procura o texto em qualquer posição, não a palavra inteira. Pra busca exata dentro de uma lista de hashtags separadas por vírgula, o jeito mais seguro é o que vem na próxima seção: separar as hashtags em linhas próprias antes de comparar.

## 8. Agrupamento por hashtag: o núcleo da aula

Cada post pode ter várias hashtags, todas juntas numa única string separada por vírgula (`"eleicoes2026,politica,brasil"`). Pra analisar por hashtag, cada hashtag precisa virar sua própria linha, mesmo que isso duplique o resto dos dados daquele post. Esse processo chama **explodir** a coluna, e o Pandas tem um método com esse nome: `.explode()`.

In [26]:
# começamos só com as linhas que têm hashtag preenchida, senão não tem o que explodir
df_hashtags = df_com_visualizacoes.dropna(subset=["hashtags"]).copy()

df_hashtags["hashtags"] = df_hashtags["hashtags"].str.split(",")  # transforma a string "a,b,c" numa lista ["a", "b", "c"], dentro de cada célula

df_hashtags = df_hashtags.explode("hashtags")  # transforma cada item da lista numa linha própria, repetindo o resto das colunas

df_hashtags["hashtags"] = df_hashtags["hashtags"].str.strip().str.lower()  # tira espaço sobrando e padroniza minúsculas (senão "Futebol" e "futebol" contam como hashtags diferentes)

print(f"Linhas antes de explodir: {len(df_com_visualizacoes)}")
print(f"Linhas depois de explodir: {len(df_hashtags)}")
df_hashtags[["author", "hashtags", "likes", "taxa_engajamento"]].head(10)

Linhas antes de explodir: 640
Linhas depois de explodir: 2104


,author,hashtags,likes,taxa_engajamento
0,g1,g1,0,0.000000
0,g1,vacina,0,0.000000
0,g1,oms,0,0.000000
0,g1,saúde,0,0.000000
0,g1,g1saúde,0,0.000000
0,g1,tiktoknotícias,0,0.000000
1,g1,g1,188,0.020849
1,g1,tiktoknotícias,188,0.020849
1,g1,baixadafluminense,188,0.020849
1,g1,assaltante,188,0.020849


**O que observar:** o número de linhas aumentou, porque um post com 3 hashtags agora aparece 3 vezes, uma por hashtag. Isso é esperado e é exatamente o que precisamos: agora dá pra agrupar por hashtag de verdade, sem perder nenhuma ocorrência. O `.str.lower()` evita que a mesma hashtag contada com maiúscula e minúscula (comum em coleta real) vire duas hashtags diferentes.

In [27]:
contagem_hashtags = df_hashtags["hashtags"].value_counts()  # conta quantas vezes cada hashtag aparece depois de explodida
contagem_hashtags.head(10)  # as 10 hashtags mais frequentes na coleta

hashtags
g1                 110
tiktoknotícias      89
tiktokesportes      81
cnnbrasil           77
botafogo            71
coberturacazétv     66
original            64
futebol             61
fogonacopa          43
vamosbotafogo       42
Name: count, dtype: int64

## 9. Agrupando e calculando métricas por hashtag

Com uma linha por hashtag, `.groupby()` agrupa todas as linhas que compartilham o mesmo valor de hashtag, e `.agg()` calcula uma ou mais estatísticas dentro de cada grupo.

In [28]:
resumo_hashtags = df_hashtags.groupby("hashtags").agg(  # agrupa as linhas pela coluna "hashtags"
    qtd_posts=("id", "count"),  # conta quantos posts (linhas) cada hashtag teve
    curtidas_medias=("likes", "mean"),  # média de curtidas dos posts daquela hashtag
    engajamento_medio=("taxa_engajamento", "mean"),  # média da taxa de engajamento dos posts daquela hashtag
    plays_total=("plays", "sum"),  # soma total de visualizações dos posts daquela hashtag
).reset_index()  # reset_index() transforma a hashtag (que virou índice do agrupamento) de volta numa coluna normal

print(f"Hashtags distintas na coleta: {len(resumo_hashtags)}")
resumo_hashtags.head(15)  # amostra; a tabela completa entra na ordenação da próxima seção

Hashtags distintas na coleta: 636


,hashtags,qtd_posts,curtidas_medias,engajamento_medio,plays_total
0,100m,1,4147.000000,0.036526,116000
1,2016,1,431700.000000,0.136782,3300000
2,4x100m,1,740.000000,0.046000,17000
3,abelardodelaespriella,1,2622.000000,0.023045,117900
4,academia,2,1089.000000,0.090942,32400
5,aeroporto,2,30640.000000,0.043829,1091000
6,agendacultural,1,30.000000,0.021945,1686
7,agressão,1,9540.000000,0.048799,208200
8,agriculturaorgânica,1,8.000000,0.033613,238
9,agriculturaregenerativa,3,21.666667,0.038518,1759


**O que observar:** `qtd_posts` conta volume (quantos posts usaram a hashtag), enquanto `engajamento_medio` conta desempenho relativo. Uma hashtag pode aparecer pouco e ainda assim ter o maior engajamento médio, ou aparecer muito e ter engajamento mediano. Em coleta real, o topo por engajamento médio costuma vir cheio de hashtags com 1 post só (um vídeo que performou bem "arrasta" a média). São perguntas diferentes, de novo o mesmo aviso da seção 5.

## 10. Tabela-resumo final

Fechamos com uma tabela só, ordenada pelo que for mais relevante pra sua pergunta (aqui, engajamento médio), com os nomes de coluna já em português e prontos pra virar uma tabela de relatório.

In [29]:
tabela_final = resumo_hashtags.sort_values("engajamento_medio", ascending=False)  # ordena da hashtag com maior engajamento médio pra menor

tabela_final = tabela_final.rename(columns={  # troca os nomes técnicos das colunas por nomes mais claros pra um relatório
    "hashtags": "Hashtag",
    "qtd_posts": "Qtd. de posts",
    "curtidas_medias": "Curtidas médias",
    "engajamento_medio": "Engajamento médio",
    "plays_total": "Visualizações (total)",
})

tabela_final["Engajamento médio"] = (tabela_final["Engajamento médio"] * 100).round(1)  # converte a taxa (0.08) pra porcentagem (8.0), mais fácil de ler

print(f"Linhas na tabela-resumo: {len(tabela_final)}")
tabela_final.head(15)  # top 15 por engajamento médio; a tabela completa vai pro CSV na próxima célula

Linhas na tabela-resumo: 636


,Hashtag,Qtd. de posts,Curtidas médias,Engajamento médio,Visualizações (total)
504,rock,1,7817.000000,25.2,31900
377,major,1,7817.000000,25.2,31900
350,joiasdobairo,1,7817.000000,25.2,31900
16,alextelles,1,2746.000000,24.8,11500
634,ídolo,1,2746.000000,24.8,11500
433,ny,1,3120.000000,22.3,14400
593,uniforme,1,1198.000000,21.7,5609
359,lançamento,1,1198.000000,21.7,5609
446,oídolofica,3,2442.666667,20.1,41166
264,fyp,2,220285.500000,20.1,3334600


In [30]:
tabela_final.to_csv("dados/resumo_hashtags.csv", index=False)  # salva a tabela final num CSV novo, dentro de dados/, sem a coluna de índice numérico
print("Tabela salva em dados/resumo_hashtags.csv")

Tabela salva em dados/resumo_hashtags.csv


## 11. Quando der errado

- `FileNotFoundError` ao ler o CSV: confira se o arquivo está em `dados/exportacao.csv` e se o terminal (ou o Jupyter) foi aberto na pasta desta aula.
- `ZeroDivisionError` ou uma coluna cheia de `inf`/`NaN` depois de dividir: alguma linha tem `plays` igual a zero. É pra isso que filtramos `df["plays"] > 0` antes de calcular a taxa de engajamento, confira se esse passo não foi pulado.
- `KeyError` numa coluna: o nome da coluna não existe exatamente daquele jeito no seu CSV (maiúscula/minúscula importa, e plataformas diferentes usam nomes diferentes). Rode `df.columns` pra ver a lista exata de colunas disponíveis.
- `.explode()` não muda o número de linhas: confira se a coluna `hashtags` já foi transformada em lista com `.str.split(",")` antes de explodir; se ela ainda for texto, o `.explode()` não tem o que separar.
- Tabela de agrupamento vazia ou com só uma linha: confira se a coluna usada no `.groupby()` está mesmo com um valor por hashtag (depois do `.explode()`), e não ainda com a string original cheia de vírgulas.

## Prática: faça agora

Abra `exercicios/exercicio-05-metricas-exploracao.ipynb`. Ele é o **Projeto 2** do curso: repete essa análise de métricas e agrupamento por hashtag, agora com a sua própria coleta da Aula 4 (ou um recorte diferente dela), a partir de uma pergunta que você mesma/o define.